# Agent 1 Contact Expert 워크플로우 테스트

이 노트북은 `Contact Expert` (접촉 불량 전문가)의 독립적인 워크플로우를 순서대로 테스트합니다.

## 워크플로우 순서

1. **초기 설정** - 프로젝트 경로 및 환경 변수 설정
2. **이미지 준비** - 테스트 이미지 로드
3. **Payload 생성** - LLM 입력 형식으로 변환
4. **프롬프트 확인** (선택사항) - 현재 적용된 프롬프트 확인
5. **Contact Expert 실행** - 4단계 순차 분석 실행
   - Step 1: 위치 식별 (Location Context)
   - Step 2: 색상 분석 (Spectral Analysis)
   - Step 3: 열적 구배 분석 (Thermal Gradient)
   - Step 4: 표면 부식 분석 (Surface Erosion)
6. **결과 분석** - 단계별 결과 확인 및 시각화
7. **최종 리포트** - 종합 리포트 확인

## 사용 방법

각 셀을 순서대로 실행하세요. 특정 단계만 테스트하려면 해당 셀만 실행하면 됩니다.

In [1]:
# 1. 초기 설정 및 라이브러리 임포트
import sys
import os
from pathlib import Path
import json
import base64

# 프로젝트 루트 경로 설정
project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(project_root))

# 환경 변수 로드
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

print(f"✓ 프로젝트 루트: {project_root}")

✓ 프로젝트 루트: c:\Users\loidn\Documents\Projects\P_04_Scope


In [2]:
# 2. 테스트 이미지 준비
from src.utils import find_data_directory

try:
    data_dir = find_data_directory()
    test_image_name = "IMG_8113.jpg"
    test_image_path = Path(data_dir) / test_image_name
    
    if not test_image_path.exists():
        # 이미지가 없으면 첫 번째 가능한 이미지 사용
        image_files = list(Path(data_dir).glob("*.png")) + list(Path(data_dir).glob("*.jpg"))
        if image_files:
            test_image_path = image_files[0]
            print(f"⚠️ {test_image_name}을 찾을 수 없어 {test_image_path.name}을 사용합니다.")
        else:
            raise FileNotFoundError("테스트할 이미지가 없습니다.")
            
    print(f"✓ 테스트 이미지: {test_image_path}")
except Exception as e:
    print(f"❌ 오류: {e}")

✓ 테스트 이미지: c:\Users\loidn\Documents\Projects\P_04_Scope\data\IMG_8113.jpg


In [3]:
# 3. Payload 생성 함수
def create_payload(image_path):
    with open(image_path, 'rb') as f:
        image_data = f.read()
    
    ext = Path(image_path).suffix.lower()
    mime_type = 'image/png' if ext == '.png' else 'image/jpeg'
    image_base64 = base64.b64encode(image_data).decode('utf-8')
    
    return [
        {"text": "이미지를 분석하세요."},
        {"inline_data": {"mime_type": mime_type, "data": image_base64}}
    ]

In [4]:
# 4. 프롬프트 확인 (선택사항)
# 현재 적용된 프롬프트를 확인합니다.
# 프롬프트를 수정하려면 src/prompts/contact_expert_prompts.py 파일을 편집하거나
# 노트북에서 직접 오버라이드할 수 있습니다.

from src.prompts.contact_expert_prompts import (
    get_step1_react_prompt, 
    get_step2_react_prompt, 
    get_step3_react_prompt, 
    get_step4_react_prompt
)

print("✅ 프롬프트 로드 완료")

✅ 프롬프트 로드 완료


In [5]:
# 5-B. Step 1만 실행 (위치 식별)
# Step 1: 용융흔이 발생한 위치를 식별합니다.

from src.nodes.contact_nodes import step1_node, ContactExpertState
from src.tools.experts.expert_utils import save_bytes_to_temp_file
import time

print("=" * 60)
print("Step 1: 위치 식별 (Location Context)")
print("=" * 60)
print(f"이미지: {test_image_path.name}")
print()

# 이미지를 임시 파일로 저장
with open(test_image_path, 'rb') as f:
    image_data = f.read()
temp_image_path = save_bytes_to_temp_file(image_data)

# ContactExpertState 초기화
state = ContactExpertState(
    messages=[],
    image_path=temp_image_path,
    contact_step1_result=None,
    contact_step2_result=None,
    contact_step3_result=None,
    contact_step4_result=None
)

# 실행 시간 측정
start_time = time.time()

try:
    result_state = step1_node(state)
    elapsed_time = time.time() - start_time
    
    print("\n" + "=" * 60)
    print("✓ Step 1 완료!")
    print(f"소요 시간: {elapsed_time:.2f}초")
    print("=" * 60)
    
    # 결과 추출
    step1_result = result_state.get("contact_step1_result", {})
    updated_image_path = result_state.get("image_path", temp_image_path)
    
    # 결과 저장
    step1_result_only = step1_result
    contact_state_step1 = result_state
    
    # 결과 출력
    if step1_result:
        print("\n📊 Step 1 결과:")
        print("-" * 60)
        
        # 새로운 프롬프트 구조: feature_name, box_2d, observation_summary, confidence
        if 'feature_name' in step1_result:
            print(f"식별된 특징: {step1_result.get('feature_name', 'N/A')}")
        
        if 'box_2d' in step1_result:
            box = step1_result['box_2d']
            print(f"Bounding Box: [{box[0]}, {box[1]}, {box[2]}, {box[3]}] (ymin, xmin, ymax, xmax)")
        
        if 'observation_summary' in step1_result:
            print(f"\n정밀 감식 소견:")
            print(f"  {step1_result.get('observation_summary', 'N/A')}")
        
        if 'confidence' in step1_result:
            print(f"\n신뢰도: {step1_result.get('confidence', 'N/A')}%")
        
        # 매핑된 필드 (contact_tools.py에서 변환된 경우)
        if 'is_connection_point' in step1_result or 'location_type' in step1_result:
            print(f"\n매핑된 정보:")
            if 'is_connection_point' in step1_result:
                print(f"  - 접속점 확인: {step1_result.get('is_connection_point', 'N/A')}")
            if 'location_type' in step1_result:
                print(f"  - 위치 유형: {step1_result.get('location_type', 'N/A')}")
            if 'location_description' in step1_result:
                print(f"  - 위치 설명: {step1_result.get('location_description', 'N/A')}")
            if 'reasoning' in step1_result:
                reasoning = step1_result.get('reasoning', '')
                if len(reasoning) > 150:
                    reasoning = reasoning[:150] + "..."
                print(f"  - 판단 근거: {reasoning}")
        
        print("\n전체 JSON 결과:")
        print(json.dumps(step1_result, indent=2, ensure_ascii=False))
        
        # box_2d 좌표가 있으면 이미지에 표시
        if 'box_2d' in step1_result and step1_result['box_2d']:
            try:
                import matplotlib.pyplot as plt
                import matplotlib.patches as patches
                from PIL import Image
                
                # 이미지 로드
                img_path = updated_image_path if updated_image_path else temp_image_path
                img = Image.open(img_path)
                original_width, original_height = img.size
                
                # box_2d 좌표 추출 (0-1000 정규화 좌표)
                box = step1_result['box_2d']
                if len(box) == 4:
                    ymin, xmin, ymax, xmax = map(float, box)
                    
                    # 정규화 좌표를 픽셀 좌표로 변환
                    x = xmin / 1000 * original_width
                    y = ymin / 1000 * original_height
                    w = (xmax - xmin) / 1000 * original_width
                    h = (ymax - ymin) / 1000 * original_height
                    
                    # 시각화
                    fig, ax = plt.subplots(figsize=(12, 12))
                    ax.imshow(img)
                    ax.set_title(f"Step 1: {step1_result.get('feature_name', '위치 식별')}", 
                               fontsize=16, fontweight='bold')
                    
                    # Bounding Box 그리기
                    rect = patches.Rectangle(
                        (x, y), w, h,
                        linewidth=3,
                        edgecolor='red',
                        facecolor='none',
                        linestyle='--'
                    )
                    ax.add_patch(rect)
                    
                    # 좌표 정보 텍스트로 표시
                    info_text = f"Box: [{int(ymin)}, {int(xmin)}, {int(ymax)}, {int(xmax)}]\n"
                    info_text += f"Pixel: ({int(x)}, {int(y)}) - ({int(x+w)}, {int(y+h)})"
                    ax.text(0.02, 0.98, info_text,
                           transform=ax.transAxes,
                           fontsize=10,
                           verticalalignment='top',
                           bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
                    
                    plt.axis('off')
                    plt.tight_layout()
                    plt.show()
                    
                    print("\n✅ Bounding Box가 이미지에 표시되었습니다.")
            except Exception as e:
                print(f"\n⚠️ 시각화 중 오류 발생: {e}")
                import traceback
                traceback.print_exc()
    else:
        print("⚠️ Step 1 결과가 없습니다.")
    
    print(f"\n✅ Step 1 결과가 저장되었습니다:")
    print(f"   - step1_result_only: Step 1 결과")
    print(f"   - contact_state_step1: 다음 Step 실행용 상태")
    print(f"   - 업데이트된 이미지 경로: {updated_image_path}")
    
except Exception as e:
    print(f"\n❌ Step 1 실행 중 오류 발생: {e}")
    import traceback
    traceback.print_exc()
    step1_result_only = {}
    contact_state_step1 = None


Step 1: 위치 식별 (Location Context)
이미지: IMG_8113.jpg


==================== Agent Reasoning & Tool Execution ====================
🛠️ [Tool Call]: crop_image (Args: {'image_path': 'C:\\Users\\loidn\\AppData\\Local\\Temp\\tmp042veemo.jpg'})
   └─ 📊 [Tool Output]: 이미지 크롭 완료
- 원본 크기: 1568x1176
- 크롭된 크기: 1568x1176

🧠 [Thought]:
[
  {
    "feature_name": "단자대 접속부의 국부적 용융 및 열적 그라데이션",
    "box_2d": [
      312,
      378,
      905,
      632
    ],
    "observation_summary": "차단기 중앙



✓ Step 1 완료!
소요 시간: 21.64초

📊 Step 1 결과:
------------------------------------------------------------

전체 JSON 결과:
{
  "result_text": "[\n  {\n    \"feature_name\": \"단자대 접속부의 국부적 용융 및 열적 그라데이션\",\n    \"box_2d\": [\n      312,\n      378,\n      905,\n      632\n    ],\n    \"observation_summary\": \"차단기 중앙"
}

✅ Step 1 결과가 저장되었습니다:
   - step1_result_only: Step 1 결과
   - contact_state_step1: 다음 Step 실행용 상태
   - 업데이트된 이미지 경로: C:\Users\loidn\AppData\Local\Temp\tmp042veemo.jpg


In [ ]:
# 5-C. Step 2만 실행 (색상 분석)
# Step 2: 아산화동(Cu₂O)을 의심할 수 있는 색상 패턴을 관찰합니다.
# ⚠️ Step 1을 먼저 실행해야 합니다.

from src.nodes.contact_nodes import step2_node

print("=" * 60)
print("Step 2: 색상 분석 (Spectral Analysis)")
print("=" * 60)

# Step 1 결과 확인
if 'contact_state_step1' not in locals() or contact_state_step1 is None:
    print("⚠️ Step 1 결과가 없습니다. 먼저 5-B 셀을 실행하세요.")
    print("   또는 5-A 셀을 실행하여 전체 분석을 완료하세요.")
else:
    print(f"이미지: {contact_state_step1.get('image_path', 'N/A')}")
    print()
    
    # 실행 시간 측정
    start_time = time.time()
    
    try:
        result_state = step2_node(contact_state_step1)
        elapsed_time = time.time() - start_time
        
        print("\n" + "=" * 60)
        print("✓ Step 2 완료!")
        print(f"소요 시간: {elapsed_time:.2f}초")
        print("=" * 60)
        
        # 결과 추출
        step2_result = result_state.get("contact_step2_result", {})
        updated_image_path = result_state.get("image_path")
        
        # 결과 저장
        step2_result_only = step2_result
        contact_state_step2 = result_state
        
        # 결과 출력
        if step2_result:
            print("\n📊 Step 2 결과:")
            print("-" * 60)
            print(f"의심 색상 패턴: {step2_result.get('suspicious_color_pattern_detected', 'N/A')}")
            print(f"아산화동 의심도: {step2_result.get('cuprous_oxide_suspicion_level', 'N/A')}")
            print(f"신뢰도: {step2_result.get('confidence', 'N/A')}%")
            
            if 'color_analysis' in step2_result:
                color_analysis = step2_result['color_analysis']
                print(f"붉은색 톤: {color_analysis.get('red_tone_present', 'N/A')}")
                print(f"주황색 톤: {color_analysis.get('orange_tone_present', 'N/A')}")
            
            print("\n전체 JSON 결과:")
            print(json.dumps(step2_result, indent=2, ensure_ascii=False))
        else:
            print("⚠️ Step 2 결과가 없습니다.")
        
        print(f"\n✅ Step 2 결과가 저장되었습니다:")
        print(f"   - step2_result_only: Step 2 결과")
        print(f"   - contact_state_step2: 다음 Step 실행용 상태")
        
    except Exception as e:
        print(f"\n❌ Step 2 실행 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()
        step2_result_only = {}
        contact_state_step2 = None


In [ ]:
# 5-D. Step 3만 실행 (열적 구배 분석)
# Step 3: 열적 구배(Thermal Gradient) 패턴을 분석합니다.
# ⚠️ Step 1, 2를 먼저 실행해야 합니다.

from src.nodes.contact_nodes import step3_node

print("=" * 60)
print("Step 3: 열적 구배 분석 (Thermal Gradient)")
print("=" * 60)

# Step 2 결과 확인
if 'contact_state_step2' not in locals() or contact_state_step2 is None:
    print("⚠️ Step 2 결과가 없습니다. 먼저 5-C 셀을 실행하세요.")
    print("   또는 5-A 셀을 실행하여 전체 분석을 완료하세요.")
else:
    print(f"이미지: {contact_state_step2.get('image_path', 'N/A')}")
    print()
    
    # 실행 시간 측정
    start_time = time.time()
    
    try:
        result_state = step3_node(contact_state_step2)
        elapsed_time = time.time() - start_time
        
        print("\n" + "=" * 60)
        print("✓ Step 3 완료!")
        print(f"소요 시간: {elapsed_time:.2f}초")
        print("=" * 60)
        
        # 결과 추출
        step3_result = result_state.get("contact_step3_result", {})
        updated_image_path = result_state.get("image_path")
        
        # 결과 저장
        step3_result_only = step3_result
        contact_state_step3 = result_state
        
        # 결과 출력
        if step3_result:
            print("\n📊 Step 3 결과:")
            print("-" * 60)
            print(f"열적 구배 감지: {step3_result.get('thermal_gradient_detected', 'N/A')}")
            print(f"구배 패턴: {step3_result.get('gradient_pattern', 'N/A')}")
            print(f"열원 위치: {step3_result.get('heat_source_location', 'N/A')}")
            print(f"열 전파 방향: {step3_result.get('heat_propagation_direction', 'N/A')}")
            print(f"신뢰도: {step3_result.get('confidence', 'N/A')}%")
            
            print("\n전체 JSON 결과:")
            print(json.dumps(step3_result, indent=2, ensure_ascii=False))
        else:
            print("⚠️ Step 3 결과가 없습니다.")
        
        print(f"\n✅ Step 3 결과가 저장되었습니다:")
        print(f"   - step3_result_only: Step 3 결과")
        print(f"   - contact_state_step3: 다음 Step 실행용 상태")
        
    except Exception as e:
        print(f"\n❌ Step 3 실행 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()
        step3_result_only = {}
        contact_state_step3 = None


In [ ]:
# 5-E. Step 4만 실행 (표면 부식 분석)
# Step 4: 금속 표면의 전기적 부식 흔적을 분석합니다.
# ⚠️ Step 1, 2, 3을 먼저 실행해야 합니다.

from src.nodes.contact_nodes import step4_node

print("=" * 60)
print("Step 4: 표면 부식 분석 (Surface Erosion)")
print("=" * 60)

# Step 3 결과 확인
if 'contact_state_step3' not in locals() or contact_state_step3 is None:
    print("⚠️ Step 3 결과가 없습니다. 먼저 5-D 셀을 실행하세요.")
    print("   또는 5-A 셀을 실행하여 전체 분석을 완료하세요.")
else:
    print(f"이미지: {contact_state_step3.get('image_path', 'N/A')}")
    print()
    
    # 실행 시간 측정
    start_time = time.time()
    
    try:
        result_state = step4_node(contact_state_step3)
        elapsed_time = time.time() - start_time
        
        print("\n" + "=" * 60)
        print("✓ Step 4 완료!")
        print(f"소요 시간: {elapsed_time:.2f}초")
        print("=" * 60)
        
        # 결과 추출
        step4_result = result_state.get("contact_step4_result", {})
        updated_image_path = result_state.get("image_path")
        
        # 결과 저장
        step4_result_only = step4_result
        contact_state_step4 = result_state
        
        # 결과 출력
        if step4_result:
            print("\n📊 Step 4 결과:")
            print("-" * 60)
            print(f"전기적 부식 감지: {step4_result.get('electrical_erosion_detected', 'N/A')}")
            print(f"표면 질감: {step4_result.get('surface_texture', 'N/A')}")
            print(f"곰보 자국 감지: {step4_result.get('pitting_detected', 'N/A')}")
            print(f"곰보 자국 설명: {step4_result.get('pitting_description', 'N/A')[:100]}...")
            print(f"신뢰도: {step4_result.get('confidence', 'N/A')}%")
            
            print("\n전체 JSON 결과:")
            print(json.dumps(step4_result, indent=2, ensure_ascii=False))
        else:
            print("⚠️ Step 4 결과가 없습니다.")
        
        print(f"\n✅ Step 4 결과가 저장되었습니다:")
        print(f"   - step4_result_only: Step 4 결과")
        print(f"   - contact_state_step4: 최종 상태")
        
        # 모든 Step 완료 후 종합 결과 생성
        print("\n" + "=" * 60)
        print("🎉 모든 Step 완료!")
        print("=" * 60)
        print("다음 셀에서 종합 리포트를 생성할 수 있습니다.")
        
    except Exception as e:
        print(f"\n❌ Step 4 실행 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()
        step4_result_only = {}
        contact_state_step4 = None


In [ ]:
# 5-F. Step별 실행 결과 종합 리포트 생성
# Step별로 개별 실행한 경우, 결과를 종합하여 리포트를 생성합니다.

from src.tools.experts.contact_tools import (
    calculate_confidence_score,
    collect_evidence,
    generate_report
)

print("=" * 60)
print("Step별 실행 결과 종합 리포트 생성")
print("=" * 60)

# Step별 결과 확인
step1 = step1_result_only if 'step1_result_only' in locals() else {}
step2 = step2_result_only if 'step2_result_only' in locals() else {}
step3 = step3_result_only if 'step3_result_only' in locals() else {}
step4 = step4_result_only if 'step4_result_only' in locals() else {}

if not any([step1, step2, step3, step4]):
    print("⚠️ Step별 실행 결과가 없습니다.")
    print("   5-B ~ 5-E 셀을 실행하거나, 5-A 셀을 실행하여 전체 분석을 완료하세요.")
else:
    print("\n수집된 Step 결과:")
    print(f"  - Step 1: {'✓' if step1 else '✗'}")
    print(f"  - Step 2: {'✓' if step2 else '✗'}")
    print(f"  - Step 3: {'✓' if step3 else '✗'}")
    print(f"  - Step 4: {'✓' if step4 else '✗'}")
    print()
    
    try:
        # 신뢰도 점수 계산
        confidence_score = calculate_confidence_score(step1, step2, step3, step4)
        print(f"📊 계산된 신뢰도 점수: {confidence_score}%")
        
        # 증거 수집
        evidence = collect_evidence(step1, step2, step3, step4)
        print(f"📋 수집된 증거: {len(evidence)}개")
        
        # 리포트 생성
        report = generate_report(step1, step2, step3, step4, confidence_score, evidence)
        
        print("\n" + "=" * 60)
        print("종합 리포트")
        print("=" * 60)
        print(report)
        
        # 결과 저장
        contact_expert_result_combined = {
            "expert_reports": [report],
            "expert_analysis_results": {
                "contact": {
                    "step1": step1,
                    "step2": step2,
                    "step3": step3,
                    "step4": step4
                }
            },
            "expert_confidence_scores": {"contact": confidence_score},
            "expert_evidence": {"contact": evidence}
        }
        
        contact_analysis_results = contact_expert_result_combined["expert_analysis_results"]["contact"]
        
        print("\n✅ 종합 리포트가 생성되었습니다:")
        print("   - contact_expert_result_combined: 전체 결과")
        print("   - contact_analysis_results: 단계별 분석 결과")
        
    except Exception as e:
        print(f"\n❌ 리포트 생성 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()


In [ ]:
# 6. 단계별 상세 결과 분석

if 'contact_analysis_results' in locals() and contact_analysis_results:
    print("=" * 60)
    print("단계별 상세 분석 결과")
    print("=" * 60)
    
    # Step 1: 위치 식별
    step1 = contact_analysis_results.get('step1', {})
    if step1:
        print("\n[Step 1] 위치 식별 결과:")
        print("-" * 60)
        
        # 새로운 프롬프트 구조: feature_name, box_2d, observation_summary, confidence
        if 'feature_name' in step1:
            print(f"식별된 특징: {step1.get('feature_name', 'N/A')}")
        
        if 'box_2d' in step1:
            box = step1['box_2d']
            print(f"Bounding Box: [{box[0]}, {box[1]}, {box[2]}, {box[3]}] (ymin, xmin, ymax, xmax)")
        
        if 'observation_summary' in step1:
            print(f"\n정밀 감식 소견:")
            print(f"  {step1.get('observation_summary', 'N/A')}")
        
        if 'confidence' in step1:
            print(f"\n신뢰도: {step1.get('confidence', 'N/A')}%")
        
        # 매핑된 필드 (contact_tools.py에서 변환된 경우)
        if 'is_connection_point' in step1 or 'location_type' in step1:
            print(f"\n매핑된 정보:")
            if 'is_connection_point' in step1:
                print(f"  - 접속점 확인: {step1.get('is_connection_point', 'N/A')}")
            if 'location_type' in step1:
                print(f"  - 위치 유형: {step1.get('location_type', 'N/A')}")
            if 'location_description' in step1:
                print(f"  - 위치 설명: {step1.get('location_description', 'N/A')}")
            if 'reasoning' in step1:
                reasoning = step1.get('reasoning', '')
                if len(reasoning) > 200:
                    reasoning = reasoning[:200] + "..."
                print(f"  - 판단 근거: {reasoning}")
    
    # Step 2: 색상 분석
    step2 = contact_analysis_results.get('step2', {})
    if step2:
        print("\n[Step 2] 색상 분석 결과:")
        print("-" * 60)
        print(f"의심 색상 패턴: {step2.get('suspicious_color_pattern_detected', 'N/A')}")
        print(f"아산화동 의심도: {step2.get('cuprous_oxide_suspicion_level', 'N/A')}")
        print(f"신뢰도: {step2.get('confidence', 'N/A')}%")
        if 'color_analysis' in step2:
            color_analysis = step2['color_analysis']
            print(f"붉은색 톤: {color_analysis.get('red_tone_present', 'N/A')}")
            print(f"주황색 톤: {color_analysis.get('orange_tone_present', 'N/A')}")
    
    # Step 3: 열적 구배 분석
    step3 = contact_analysis_results.get('step3', {})
    if step3:
        print("\n[Step 3] 열적 구배 분석 결과:")
        print("-" * 60)
        print(f"열적 구배 감지: {step3.get('thermal_gradient_detected', 'N/A')}")
        print(f"구배 패턴: {step3.get('gradient_pattern', 'N/A')}")
        print(f"열원 위치: {step3.get('heat_source_location', 'N/A')}")
        print(f"신뢰도: {step3.get('confidence', 'N/A')}%")
    
    # Step 4: 표면 부식 분석
    step4 = contact_analysis_results.get('step4', {})
    if step4:
        print("\n[Step 4] 표면 부식 분석 결과:")
        print("-" * 60)
        print(f"전기적 부식 감지: {step4.get('electrical_erosion_detected', 'N/A')}")
        print(f"표면 질감: {step4.get('surface_texture', 'N/A')}")
        print(f"곰보 자국 감지: {step4.get('pitting_detected', 'N/A')}")
        print(f"신뢰도: {step4.get('confidence', 'N/A')}%")
    
    # 전체 JSON 출력
    print("\n" + "=" * 60)
    print("전체 JSON 결과 (보기)")
    print("=" * 60)
    print(json.dumps(contact_analysis_results, indent=2, ensure_ascii=False))
    
else:
    print("⚠️ 분석 결과가 없습니다. 먼저 5번 셀을 실행하세요.")

In [ ]:
    # Step 1: 위치 식별
    step1 = contact_analysis_results.get('step1', {})
    if step1:
        print("\n[Step 1] 위치 식별 결과:")
        print("-" * 60)
        print(f"접속점 확인: {step1.get('is_connection_point', 'N/A')}")
        print(f"위치 유형: {step1.get('location_type', 'N/A')}")
        print(f"위치 설명: {step1.get('location_description', 'N/A')}")
        print(f"이미지 품질: {step1.get('image_quality', 'N/A')}")
        print(f"신뢰도: {step1.get('confidence', 'N/A')}%")
        if 'reasoning' in step1:
            print(f"판단 근거: {step1['reasoning'][:200]}...")

In [ ]:
        # Step 2: 색상 분석
    step2 = contact_analysis_results.get('step2', {})
    if step2:
        print("\n[Step 2] 색상 분석 결과:")
        print("-" * 60)
        print(f"의심 색상 패턴: {step2.get('suspicious_color_pattern_detected', 'N/A')}")
        print(f"아산화동 의심도: {step2.get('cuprous_oxide_suspicion_level', 'N/A')}")
        print(f"신뢰도: {step2.get('confidence', 'N/A')}%")
        if 'color_analysis' in step2:
            color_analysis = step2['color_analysis']
            print(f"붉은색 톤: {color_analysis.get('red_tone_present', 'N/A')}")
            print(f"주황색 톤: {color_analysis.get('orange_tone_present', 'N/A')}")

In [ ]:
    # Step 3: 열적 구배 분석
    step3 = contact_analysis_results.get('step3', {})
    if step3:
        print("\n[Step 3] 열적 구배 분석 결과:")
        print("-" * 60)
        print(f"열적 구배 감지: {step3.get('thermal_gradient_detected', 'N/A')}")
        print(f"구배 패턴: {step3.get('gradient_pattern', 'N/A')}")
        print(f"열원 위치: {step3.get('heat_source_location', 'N/A')}")
        print(f"신뢰도: {step3.get('confidence', 'N/A')}%")

In [ ]:
    # Step 4: 표면 부식 분석
    step4 = contact_analysis_results.get('step4', {})
    if step4:
        print("\n[Step 4] 표면 부식 분석 결과:")
        print("-" * 60)
        print(f"전기적 부식 감지: {step4.get('electrical_erosion_detected', 'N/A')}")
        print(f"표면 질감: {step4.get('surface_texture', 'N/A')}")
        print(f"곰보 자국 감지: {step4.get('pitting_detected', 'N/A')}")
        print(f"신뢰도: {step4.get('confidence', 'N/A')}%")

In [ ]:
# 7. 결과 시각화 (Bounding Box 표시)
# 분석 결과에 포함된 Bounding Box를 이미지 위에 표시합니다.

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def draw_bounding_boxes(image_path, analysis_results):
    """
    분석 결과에 포함된 Bounding Box를 이미지 위에 표시합니다.
    """
    try:
        # 이미지 로드
        img = Image.open(image_path)
        original_width, original_height = img.size
        
        fig, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(img)
        ax.set_title("Contact Expert - AI Detected Regions", fontsize=16, fontweight='bold')
        
        colors = {
            'step1': ('red', 'Location'),
            'step2': ('orange', 'Color/Oxidation'),
            'step3': ('blue', 'Thermal Gradient'),
            'step4': ('green', 'Surface Erosion')
        }
        
        legend_patches = []
        boxes_found = False
        
        # 각 단계별 결과 시각화
        for step_key, (color, label) in colors.items():
            step_result = analysis_results.get(step_key, {})
            
            # bboxes 추출 시도
            bboxes = step_result.get('bboxes', [])
            
            # 하위 필드에서도 탐색
            if not bboxes:
                if 'visual_evidence' in step_result:
                    bboxes = step_result['visual_evidence'].get('bboxes', [])
                elif 'suspected_origin_box_2d' in step_result:
                    # Step 1의 경우 suspected_origin_box_2d 사용
                    box = step_result.get('suspected_origin_box_2d')
                    if box and len(box) == 4:
                        bboxes = [box]
            
            if bboxes:
                boxes_found = True
                legend_patches.append(patches.Patch(color=color, label=f"{label} ({len(bboxes)})"))
                
                for box in bboxes:
                    # 0-1000 정규화 좌표 -> 픽셀 좌표 변환
                    # box format: [ymin, xmin, ymax, xmax]
                    if len(box) == 4:
                        try:
                            ymin, xmin, ymax, xmax = map(float, box)
                            
                            x = xmin / 1000 * original_width
                            y = ymin / 1000 * original_height
                            w = (xmax - xmin) / 1000 * original_width
                            h = (ymax - ymin) / 1000 * original_height
                            
                            # 사각형 그리기
                            rect = patches.Rectangle(
                                (x, y), w, h, 
                                linewidth=3, 
                                edgecolor=color, 
                                facecolor='none',
                                linestyle='--'
                            )
                            ax.add_patch(rect)
                        except (ValueError, TypeError) as e:
                            print(f"⚠️ {step_key}의 좌표 파싱 오류: {e}")

        if legend_patches:
            ax.legend(handles=legend_patches, loc='upper right', fontsize=12)
        else:
            ax.text(0.5, 0.5, 'No Bounding Boxes Detected', 
                   transform=ax.transAxes, 
                   ha='center', va='center',
                   fontsize=14, color='gray')
            
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        
        if not boxes_found:
            print("ℹ️ 검출된 Bounding Box 정보가 없습니다.")
            print("   (프롬프트에서 bboxes 필드를 요구하지 않았거나, AI가 좌표를 제공하지 않았을 수 있습니다)")
            
    except Exception as e:
        print(f"❌ 시각화 중 오류 발생: {e}")
        import traceback
        traceback.print_exc()

# 시각화 실행
if 'test_image_path' in locals() and 'contact_analysis_results' in locals():
    if contact_analysis_results:
        draw_bounding_boxes(test_image_path, contact_analysis_results)
    else:
        print("⚠️ 분석 결과가 없습니다. 먼저 5번 셀을 실행하세요.")
else:
    print("⚠️ 테스트 이미지 경로 또는 분석 결과가 없습니다.")


In [ ]:
# 8. 증거 및 신뢰도 상세 분석

if 'contact_expert_result' in locals() and contact_expert_result:
    print("=" * 60)
    print("증거 및 신뢰도 분석")
    print("=" * 60)
    
    # 증거 수집
    evidence = contact_expert_result.get("expert_evidence", {}).get("contact", [])
    if evidence:
        print("\n📋 수집된 증거:")
        print("-" * 60)
        for i, ev in enumerate(evidence, 1):
            step = ev.get('step', 'N/A')
            evidence_text = ev.get('evidence', 'N/A')
            print(f"{i}. [Step {step}] {evidence_text}")
    else:
        print("\n⚠️ 수집된 증거가 없습니다.")
    
    # 신뢰도 점수
    confidence = contact_expert_result.get("expert_confidence_scores", {}).get("contact", 0)
    print(f"\n📊 최종 신뢰도 점수: {confidence}%")
    
    if confidence >= 80:
        print("   → 높은 신뢰도 (High Confidence)")
    elif confidence >= 60:
        print("   → 중간 신뢰도 (Medium Confidence)")
    else:
        print("   → 낮은 신뢰도 (Low Confidence)")
    
    # 에러 확인
    errors = contact_expert_result.get("errors", [])
    if errors:
        print(f"\n⚠️ 경고/에러 ({len(errors)}개):")
        for error in errors:
            print(f"   - {error}")
    else:
        print("\n✅ 에러 없음")
        
else:
    print("⚠️ 분석 결과가 없습니다. 먼저 5번 셀을 실행하세요.")
